In [1]:
from cst_utils import set_seeds

# Manual assessment for TimesFM deterministic behavior with random states

In [3]:
import os
import random
import numpy as np
import pytest

In [5]:
set_seeds()

In [4]:
def _forecast_once(model, segments, horizon: int):
    return model.forecast(segments, horizon=horizon)

In [ ]:

@pytest.mark.parametrize("horizon", [30])
def test_timesfm_is_deterministic_over_same_segments(timesfm_model, horizon):

    segments = [
        np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32),
        np.array([10.0, 9.5, 9.0, 8.5, 8.0], dtype=np.float32),
    ]

    # ---- Run 30 times ----
    outputs = []
    for _ in range(30):
        yhat = _forecast_once(timesfm_model, segments, horizon=horizon)
        
        yhat_np = np.asarray(yhat, dtype=np.float32)
        outputs.append(yhat_np)

    # ---- Assert all equal to the first run ----
    ref = outputs[0]
    for i, out in enumerate(outputs[1:], start=1):
        assert out.shape == ref.shape, f"Run {i}: shape mismatch {out.shape} vs {ref.shape}"
        # Strongest check (bitwise):
        assert np.array_equal(out, ref), f"Run {i}: predictions differ"

In [6]:
model = "google/timesfm-1.0-200m-pytorch"
device = 'cpu'

In [13]:
import timesfm

In [19]:
# For Torch
tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=32,
          horizon_len=128,
      ),
      checkpoint=timesfm.TimesFmCheckpoint(
          huggingface_repo_id="google/timesfm-1.0-200m-pytorch"),
  )

Fetching 3 files: 100%|██████████| 3/3 [01:07<00:00, 22.57s/it]


In [23]:
import numpy as np
forecast_input = [
    np.sin(np.linspace(0, 20, 100)),
    np.sin(np.linspace(0, 20, 200)),
    np.sin(np.linspace(0, 20, 400)),
]
frequency_input = [0, 1, 2]

In [ ]:
import torch

# -----------------------------
# Run forecast 30 times
# -----------------------------
point_forecasts = []
quantile_forecasts = []

for i in range(30):
    with torch.no_grad():
        point_fcst, quantile_fcst = tfm.forecast(
            forecast_input,
            freq=frequency_input,
        )

    # Normalize outputs to stable NumPy arrays
    point_fcst = np.asarray(point_fcst, dtype=np.float32)
    quantile_fcst = np.asarray(quantile_fcst, dtype=np.float32)

    point_forecasts.append(point_fcst)
    quantile_forecasts.append(quantile_fcst)

    print(f"Run {i + 1} completed.")

Run 1 completed.
Run 2 completed.
Run 3 completed.
Run 4 completed.
Run 5 completed.
Run 6 completed.
Run 7 completed.
Run 8 completed.
Run 9 completed.
Run 10 completed.
Run 11 completed.
Run 12 completed.
Run 13 completed.
Run 14 completed.
Run 15 completed.
Run 16 completed.
Run 17 completed.
Run 18 completed.
Run 19 completed.
Run 20 completed.
Run 21 completed.
Run 22 completed.
Run 23 completed.
Run 24 completed.
Run 25 completed.
Run 26 completed.
Run 27 completed.
Run 28 completed.
Run 29 completed.
Run 30 completed.


In [ ]:

ref_point = point_forecasts[0]
ref_quantile = quantile_forecasts[0]

for i in range(1, 30):
    assert point_forecasts[i].shape == ref_point.shape, (
        f"Point forecast shape mismatch at run {i + 1}"
    )
    assert quantile_forecasts[i].shape == ref_quantile.shape, (
        f"Quantile forecast shape mismatch at run {i + 1}"
    )

    assert np.array_equal(point_forecasts[i], ref_point), (
        f"Point forecast differs at run {i + 1}"
    )
    assert np.array_equal(quantile_forecasts[i], ref_quantile), (
        f"Quantile forecast differs at run {i + 1}"
    )

print("✅ Determinism test PASSED: all 30 forecasts are identical.")

✅ Determinism test PASSED: all 30 forecasts are identical.
